In [5]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import joblib
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

from riskgate import config as cfg
from riskgate.features import filter_to_binary_outcomes
from riskgate.modeling import build_pipeline, fit_calibrated_model

In [6]:
notebook_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "notebook_model_train.csv")
raw_df = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "lc_trainingset.csv")

# Keep only binary outcomes in raw data and map to 0/1
raw_df = filter_to_binary_outcomes(raw_df)

print("Notebook-style dataframe shape:", notebook_df.shape)
print("Raw deployed-style dataframe shape:", raw_df.shape)

print("\nNotebook columns:")
print(notebook_df.columns.tolist())

print("\nRaw columns:")
print(raw_df.columns.tolist())

Notebook-style dataframe shape: (316824, 15)
Raw deployed-style dataframe shape: (316824, 29)

Notebook columns:
['zip_code', 'sub_grade', 'term', 'application_type', 'home_ownership', 'int_rate', 'title', 'mort_acc', 'dti', 'annual_inc', 'verification_status', 'emp_title', 'purpose', 'loan_amnt', 'loan_status']

Raw columns:
['id', 'loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'purpose', 'title', 'dti', 'earliest_cr_line', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'application_type', 'mort_acc', 'pub_rec_bankruptcies', 'address', 'loan_status', 'target_default']


In [7]:
notebook_features = [
    "zip_code",
    "sub_grade",
    "term",
    "application_type",
    "home_ownership",
    "int_rate",
    "title",
    "mort_acc",
    "dti",
    "annual_inc",
    "verification_status",
    "emp_title",
    "purpose",
    "loan_amnt",
]

# Notebook-style processed feature matrix
X_nb_full = notebook_df[notebook_features].copy()
y_nb_full = notebook_df["loan_status"].astype(int).copy()

# Raw deployed-style matrix
X_raw_full = raw_df.drop(columns=[cfg.TARGET_COL, cfg.TARGET_BINARY_COL]).copy()
y_raw_full = raw_df[cfg.TARGET_BINARY_COL].astype(int).copy()

# Shared row-index split to keep notebook vs deployed comparable
row_idx = np.arange(len(notebook_df))

train_idx, test_idx = train_test_split(
    row_idx,
    test_size=0.30,
    random_state=0,
    shuffle=True
)

print("Train rows:", len(train_idx))
print("Test rows:", len(test_idx))

Train rows: 221776
Test rows: 95048


In [8]:
X_train_nb = X_nb_full.iloc[train_idx].reset_index(drop=True)
X_test_nb  = X_nb_full.iloc[test_idx].reset_index(drop=True)
y_train_nb = y_nb_full.iloc[train_idx].reset_index(drop=True)
y_test_nb  = y_nb_full.iloc[test_idx].reset_index(drop=True)

X_train_raw = X_raw_full.iloc[train_idx].reset_index(drop=True)
X_test_raw  = X_raw_full.iloc[test_idx].reset_index(drop=True)
y_train_raw = y_raw_full.iloc[train_idx].reset_index(drop=True)
y_test_raw  = y_raw_full.iloc[test_idx].reset_index(drop=True)

print("Notebook train/test:", X_train_nb.shape, X_test_nb.shape)
print("Raw train/test:", X_train_raw.shape, X_test_raw.shape)

Notebook train/test: (221776, 14) (95048, 14)
Raw train/test: (221776, 27) (95048, 27)


In [9]:
notebook_model = xgb.XGBClassifier(
    random_state=0,
    reg_lambda=2,
    reg_alpha=5,
    max_depth=4,
    learning_rate=0.26,
    gamma=0.4,
    colsample_bytree=1.0,
    eval_metric="logloss"
)

notebook_model.fit(X_train_nb, y_train_nb)
p_nb = notebook_model.predict_proba(X_test_nb)[:, 1]

print("Notebook baseline")
print("ROC-AUC :", roc_auc_score(y_test_nb, p_nb))
print("AP      :", average_precision_score(y_test_nb, p_nb))
print("Brier   :", brier_score_loss(y_test_nb, p_nb))

Notebook baseline
ROC-AUC : 0.9097792632943689
AP      : 0.7852013757526997
Brier   : 0.08073929541057497


In [10]:
deployed_uncal = build_pipeline("xgb")
deployed_uncal.fit(X_train_raw, y_train_raw)
p_dep_uncal = deployed_uncal.predict_proba(X_test_raw)[:, 1]

print("Deployed pipeline (uncalibrated)")
print("ROC-AUC :", roc_auc_score(y_test_raw, p_dep_uncal))
print("AP      :", average_precision_score(y_test_raw, p_dep_uncal))
print("Brier   :", brier_score_loss(y_test_raw, p_dep_uncal))

Deployed pipeline (uncalibrated)
ROC-AUC : 0.9094379165894273
AP      : 0.7844744628361087
Brier   : 0.08083348794106163


In [11]:
deployed_cal = fit_calibrated_model(build_pipeline("xgb"), X_train_raw, y_train_raw)
p_dep_cal = deployed_cal.predict_proba(X_test_raw)[:, 1]

print("Deployed pipeline (calibrated)")
print("ROC-AUC :", roc_auc_score(y_test_raw, p_dep_cal))
print("AP      :", average_precision_score(y_test_raw, p_dep_cal))
print("Brier   :", brier_score_loss(y_test_raw, p_dep_cal))

Deployed pipeline (calibrated)
ROC-AUC : 0.9098485934187939
AP      : 0.7853030753415191
Brier   : 0.08281415876787389


In [12]:
def summarize_model(y_true, p, name):
    return {
        "model": name,
        "roc_auc": roc_auc_score(y_true, p),
        "average_precision": average_precision_score(y_true, p),
        "brier": brier_score_loss(y_true, p),
        "pd_mean": float(np.mean(p)),
        "pd_std": float(np.std(p)),
    }

comparison_df = pd.DataFrame([
    summarize_model(y_test_nb, p_nb, "Notebook baseline XGB"),
    summarize_model(y_test_raw, p_dep_uncal, "Deployed pipeline XGB (uncalibrated)"),
    summarize_model(y_test_raw, p_dep_cal, "Deployed pipeline XGB (calibrated)"),
])

comparison_df

,model,roc_auc,average_precision,brier,pd_mean,pd_std
0,Notebook baseline XGB,0.909779,0.785201,0.080739,0.197368,0.278270
1,Deployed pipeline XGB (uncalibrated),0.909438,0.784474,0.080833,0.197328,0.277277
2,Deployed pipeline XGB (calibrated),0.909849,0.785303,0.082814,0.197694,0.281595


In [13]:
def assign_decision(pd_scores, t_low=0.10, t_high=0.28):
    return np.where(
        pd_scores < t_low,
        "approve",
        np.where(pd_scores < t_high, "review", "reject")
    )

def decision_counts(p, t_low=0.10, t_high=0.28):
    dec = pd.Series(assign_decision(p, t_low=t_low, t_high=t_high))
    vc = dec.value_counts()
    return {
        "approve": int(vc.get("approve", 0)),
        "review": int(vc.get("review", 0)),
        "reject": int(vc.get("reject", 0)),
    }

policy_df = pd.DataFrame([
    {"model": "Notebook baseline XGB", **decision_counts(p_nb, cfg.FROZEN_T_LOW, cfg.FROZEN_T_HIGH)},
    {"model": "Deployed pipeline XGB (uncalibrated)", **decision_counts(p_dep_uncal, cfg.FROZEN_T_LOW, cfg.FROZEN_T_HIGH)},
    {"model": "Deployed pipeline XGB (calibrated)", **decision_counts(p_dep_cal, cfg.FROZEN_T_LOW, cfg.FROZEN_T_HIGH)},
])

policy_df

,model,approve,review,reject
0,Notebook baseline XGB,46904,27404,20740
1,Deployed pipeline XGB (uncalibrated),46011,28540,20497
2,Deployed pipeline XGB (calibrated),56878,19578,18592


In [14]:
corr_nb_dep_uncal = np.corrcoef(p_nb, p_dep_uncal)[0, 1]
corr_nb_dep_cal = np.corrcoef(p_nb, p_dep_cal)[0, 1]

dec_nb = assign_decision(p_nb, cfg.FROZEN_T_LOW, cfg.FROZEN_T_HIGH)
dec_dep_uncal = assign_decision(p_dep_uncal, cfg.FROZEN_T_LOW, cfg.FROZEN_T_HIGH)
dec_dep_cal = assign_decision(p_dep_cal, cfg.FROZEN_T_LOW, cfg.FROZEN_T_HIGH)

agreement_uncal = np.mean(dec_nb == dec_dep_uncal)
agreement_cal = np.mean(dec_nb == dec_dep_cal)

print("Correlation: notebook vs deployed uncalibrated =", round(corr_nb_dep_uncal, 4))
print("Correlation: notebook vs deployed calibrated   =", round(corr_nb_dep_cal, 4))
print("Decision agreement: notebook vs deployed uncalibrated =", round(agreement_uncal, 4))
print("Decision agreement: notebook vs deployed calibrated   =", round(agreement_cal, 4))

Correlation: notebook vs deployed uncalibrated = 0.9923
Correlation: notebook vs deployed calibrated   = 0.9788
Decision agreement: notebook vs deployed uncalibrated = 0.9047
Decision agreement: notebook vs deployed calibrated   = 0.8465


In [15]:
bundle = joblib.load(PROJECT_ROOT / "artifacts" / "final_model_bundle.joblib")
bundle_model = bundle["model"]
bundle_thresholds = bundle["thresholds"]

p_bundle = bundle_model.predict_proba(X_test_raw)[:, 1]

bundle_summary = pd.DataFrame([{
    "model": "Saved final bundle",
    "roc_auc": roc_auc_score(y_test_raw, p_bundle),
    "average_precision": average_precision_score(y_test_raw, p_bundle),
    "brier": brier_score_loss(y_test_raw, p_bundle),
    "pd_mean": float(np.mean(p_bundle)),
    "pd_std": float(np.std(p_bundle)),
    "approve": int((p_bundle < bundle_thresholds["t_low"]).sum()),
    "review": int(((p_bundle >= bundle_thresholds["t_low"]) & (p_bundle < bundle_thresholds["t_high"])).sum()),
    "reject": int((p_bundle >= bundle_thresholds["t_high"]).sum()),
}])

bundle_summary

,model,roc_auc,average_precision,brier,pd_mean,pd_std,approve,review,reject
0,Saved final bundle,0.913253,0.792037,0.079663,0.19737,0.277472,46248,28255,20545


In [16]:
import joblib
bundle = joblib.load("artifacts/final_model_bundle.joblib")
print(type(bundle["model"]))
print(type(bundle["model"]).__name__)
print(bundle["metadata"].get("deployment_model_variant"))

<class 'sklearn.pipeline.Pipeline'>
Pipeline
uncalibrated_pipeline


In [17]:
import joblib
import numpy as np
import pandas as pd

bundle = joblib.load("artifacts/final_model_bundle.joblib")
model = bundle["model"]
t_low = bundle["thresholds"]["t_low"]
t_high = bundle["thresholds"]["t_high"]

df = pd.read_csv("data/scored/new_applications_to_score.csv")
p = model.predict_proba(df)[:, 1]

decision = np.where(
    p < t_low,
    "approve",
    np.where(p < t_high, "review", "reject")
)

loan_amnt = pd.to_numeric(df.get("loan_amnt", 0), errors="coerce").fillna(0.0)
risk_proxy_nonreject = float((p[decision != "reject"] * loan_amnt[decision != "reject"]).sum())

summary = {
    "t_low": t_low,
    "t_high": t_high,
    "rows": len(df),
    "approve": int((decision == "approve").sum()),
    "review": int((decision == "review").sum()),
    "reject": int((decision == "reject").sum()),
    "approve_rate": float((decision == "approve").mean()),
    "review_rate": float((decision == "review").mean()),
    "reject_rate": float((decision == "reject").mean()),
    "avg_pd_overall": float(p.mean()),
    "avg_pd_approve": float(p[decision == "approve"].mean()) if (decision == "approve").any() else np.nan,
    "avg_pd_review": float(p[decision == "review"].mean()) if (decision == "review").any() else np.nan,
    "avg_pd_reject": float(p[decision == "reject"].mean()) if (decision == "reject").any() else np.nan,
    "risk_proxy_nonreject": risk_proxy_nonreject,
}

summary

{'t_low': 0.1,
 't_high': 0.28,
 'rows': 78237,
 'approve': 38175,
 'review': 23202,
 'reject': 16860,
 'approve_rate': 0.48794048851566396,
 'review_rate': 0.29656045093753597,
 'reject_rate': 0.2154990605468001,
 'avg_pd_overall': 0.1963547021150589,
 'avg_pd_approve': 0.01908074878156185,
 'avg_pd_review': 0.17637573182582855,
 'avg_pd_reject': 0.625238835811615,
 'risk_proxy_nonreject': 65154148.18533119}

In [18]:
import joblib
import pandas as pd
from webapp.assistant import build_scenario_grid_from_scores, recommend_thresholds

bundle = joblib.load("artifacts/final_model_bundle.joblib")
model = bundle["model"]

df = pd.read_csv("data/scored/new_applications_to_score.csv")
p = model.predict_proba(df)[:, 1]

base_scored_df = pd.DataFrame({
    "pd_score": p,
    "loan_amnt": pd.to_numeric(df["loan_amnt"], errors="coerce").fillna(0.0)
})

grid = build_scenario_grid_from_scores(base_scored_df)

for goal in ["Balanced", "Growth", "Conservative", "Operations-first"]:
    rec = recommend_thresholds(
        grid=grid,
        goal=goal,
        max_review_rate=0.20,
        min_approve_rate=0.50
    )
    print("\nGOAL:", goal)
    print(rec[[
        "t_low", "t_high",
        "approve_rate", "review_rate", "reject_rate",
        "avg_pd_approve", "avg_pd_review",
        "risk_proxy_nonreject", "risk_proxy_norm", "score"
    ]].head(1))


GOAL: Balanced
    t_low  t_high  approve_rate  review_rate  reject_rate  avg_pd_approve  \
39   0.11    0.17      0.511318     0.124596     0.364086        0.023006   

    avg_pd_review  risk_proxy_nonreject  risk_proxy_norm     score  
39       0.138435          2.934291e+07         0.081464  0.453808  

GOAL: Growth
    t_low  t_high  approve_rate  review_rate  reject_rate  avg_pd_approve  \
81   0.19    0.25      0.669824     0.082697     0.247479        0.052422   

    avg_pd_review  risk_proxy_nonreject  risk_proxy_norm     score  
81        0.21823          5.537683e+07         0.427906  0.651117  

GOAL: Conservative
    t_low  t_high  approve_rate  review_rate  reject_rate  avg_pd_approve  \
39   0.11    0.17      0.511318     0.124596     0.364086        0.023006   

    avg_pd_review  risk_proxy_nonreject  risk_proxy_norm     score  
39       0.138435          2.934291e+07         0.081464 -0.047151  

GOAL: Operations-first
    t_low  t_high  approve_rate  review_rate  r

In [24]:
import importlib
import webapp.assistant as assistant
importlib.reload(assistant)

import joblib
import numpy as np
import pandas as pd
from webapp.assistant import build_scenario_grid_from_scores, recommend_thresholds

bundle = joblib.load("artifacts/final_model_bundle.joblib")
model = bundle["model"]
frozen_t_low = float(bundle["thresholds"]["t_low"])
frozen_t_high = float(bundle["thresholds"]["t_high"])

df = pd.read_csv("data/scored/new_applications_to_score.csv")
pd_scores = model.predict_proba(df)[:, 1]

base_scored_df = pd.DataFrame({
    "pd_score": pd_scores,
    "loan_amnt": pd.to_numeric(df.get("loan_amnt", 0), errors="coerce").fillna(0.0)
})

t_low_candidates = np.sort(np.unique(np.round(
    np.append(np.arange(0.05, 0.21, 0.02), [frozen_t_low]), 2
)))
t_high_candidates = np.sort(np.unique(np.round(
    np.append(np.arange(0.15, 0.41, 0.02), [frozen_t_high]), 2
)))

scenario_grid = build_scenario_grid_from_scores(
    base_scored_df,
    t_low_values=t_low_candidates,
    t_high_values=t_high_candidates,
)

rec = recommend_thresholds(
    grid=scenario_grid,
    goal="Frozen benchmark",
    max_review_rate=0.20,
    min_approve_rate=0.50,
    frozen_t_low=frozen_t_low,
    frozen_t_high=frozen_t_high,
)

print(rec[["t_low", "t_high", "approve_rate", "review_rate", "reject_rate", "score"]])

    t_low  t_high  approve_rate  review_rate  reject_rate  score
48    0.1    0.28       0.48794      0.29656     0.215499    NaN


In [25]:
import pandas as pd
import numpy as np

TEXT_UPLOAD_COLUMNS = [
    "application_id", "term", "emp_length", "grade", "sub_grade",
    "home_ownership", "verification_status", "purpose", "application_type",
    "initial_list_status", "issue_d", "earliest_cr_line", "address",
    "zip_code", "state", "addr_state",
]

def sanitize_uploaded_input(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        dtype_name = str(out[col].dtype)
        if col in TEXT_UPLOAD_COLUMNS or pd.api.types.is_string_dtype(out[col]) or pd.api.types.is_object_dtype(out[col]):
            out[col] = out[col].astype("object")
            out.loc[pd.isna(out[col]), col] = np.nan
        elif dtype_name in ["Int64", "Int32", "Int16", "Int8", "UInt64", "UInt32", "UInt16", "UInt8", "Float64", "Float32"]:
            out[col] = pd.to_numeric(out[col], errors="coerce")
        elif dtype_name == "boolean":
            out[col] = out[col].astype("object")
            out.loc[pd.isna(out[col]), col] = np.nan
    return out

dtype_map = {col: "string" for col in TEXT_UPLOAD_COLUMNS}
test_df = pd.read_csv("riskgate_randomized_customers_10000.csv", dtype=dtype_map)
test_df = sanitize_uploaded_input(test_df)

bundle = joblib.load("artifacts/final_model_bundle.joblib")
model = bundle["model"]

proba = model.predict_proba(test_df)[:, 1]
print(len(proba), proba[:5])
print(test_df.dtypes.head(10))

FileNotFoundError: [Errno 2] No such file or directory: 'riskgate_randomized_customers_10000.csv'